<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/M%C3%A1xima_Persistencia_H1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hopf Normal

Este notebook calcula la máxima persistencia de H₁ para el sistema Hopf normal con μ ∈ [-1,1] (300 valores). Usa embedding de Takens (d=2, τ=26, stride=2), Vietoris-Rips con métrica Manhattan y ruido = 0%. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence
import pandas as pd
import os

# ============================================================
# 1. INTEGRADOR RK4 (RUNGE-KUTTA DE 4to ORDEN)
# ============================================================

def rk4_step(f, y, t, dt):
    """
    Un paso del método RK4.
    f: función del sistema dinámico
    y: estado actual
    t: tiempo actual
    dt: paso de tiempo
    """
    k1 = np.asarray(f(y, t))
    k2 = np.asarray(f(y + dt/2 * k1, t + dt/2))
    k3 = np.asarray(f(y + dt/2 * k2, t + dt/2))
    k4 = np.asarray(f(y + dt * k3, t + dt))
    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

def rk4_integrate(f, y0, t):
    """
    Integra una EDO usando RK4.
    f: función del sistema
    y0: condición inicial
    t: vector de tiempos
    """
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    dt = t[1] - t[0]
    for i in range(1, len(t)):
        y[i] = rk4_step(f, y[i-1], t[i-1], dt)
    return y

# ============================================================
# 2. SIMULADOR DEL SISTEMA DE HOPF NORMAL
# ============================================================

L = 50          # Duración total de la simulación
fs = 100        # Frecuencia de muestreo
SampleSize = 2000  # Número de puntos a retener

def simulate_hopf(mu, omega=6.0, L=L, fs=fs, SampleSize=SampleSize, InitialConditions=None):
    """
    Simula el sistema de Hopf normal.
    mu: parámetro de bifurcación
    omega: frecuencia angular
    """
    if InitialConditions is None:
        InitialConditions = [0.0, 1.01]  # Condiciones iniciales [x0, y0]

    t = np.linspace(0, L, int(L * fs))  # Vector de tiempo

    def hopf_normal(state, t):
        """Ecuaciones diferenciales del sistema Hopf normal."""
        x, y = state
        r2 = x**2 + y**2
        dxdt = mu * x - omega * y - r2 * x
        dydt = omega * x + mu * y - r2 * y
        return dxdt, dydt

    states = rk4_integrate(hopf_normal, InitialConditions, t)

    # Extraer las series temporales x(t) e y(t), tomando solo los últimos SampleSize puntos
    ts = [states[:, 0][-SampleSize:], states[:, 1][-SampleSize:]]
    t = t[-SampleSize:]

    return t, ts

# ============================================================
# 3. ESTILO DE LAS GRÁFICAS
# ============================================================

REF_COLOR = "#1F3FFF"   # Azul oscuro para referencias
CHG_COLOR = "#D62728"   # Rojo oscuro para datos principales
DIAG_COLOR = "#000000"  # Negro para líneas diagonales
GRID_COLOR = "#C8C8C8"  # Gris claro para la cuadrícula
BG_COLOR = "#FFFFFF"    # Blanco para el fondo

# Configuración global de matplotlib
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.facecolor": BG_COLOR,
    "figure.facecolor": "white",
})

def style_2d_axes(ax):
    """Aplica un estilo consistente a los ejes 2D."""
    ax.grid(True, alpha=0.22, color=GRID_COLOR, linewidth=0.7)

    # Configurar bordes del gráfico
    for spine in ["left", "bottom", "top", "right"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(0.5)

    # Configurar marcas (ticks)
    ax.tick_params(axis="both", which="major", length=3, width=0.8, color="black", labelcolor="black")

    # Configurar etiquetas
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")

# ============================================================
# 4. RUIDO CONTROLADO (OPCIONAL)
# ============================================================

ruido = 0.00  # Nivel de ruido (0% = sin ruido)

def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
    """
    Agrega ruido gaussiano controlado a una serie temporal.
    mode: "absolute" o "relative" (relativo al desvío estándar)
    """
    if seed is not None:
        np.random.seed(seed)

    ts = np.asarray(ts)

    if mode == "relative":
        scale = noise_level * np.std(ts, axis=-1, keepdims=True)
    else:
        scale = noise_level

    noise = scale * np.random.randn(*ts.shape)
    return ts + noise

# ============================================================
# 5. EMBEDDING DE TAKENS
# ============================================================

dim = 2      # Dimensión del embedding
tau = 26     # Tiempo de retardo
dT = 2       # Stride (paso entre ventanas)
max_edge = 20  # Distancia máxima para el complejo de Vietoris-Rips

def getTakensEmbedding(x, dim, Tau, dT=dT):
    """
    Calcula el embedding de Takens usando giotto-tda.
    x: serie temporal
    dim: dimensión del embedding
    Tau: tiempo de retardo
    dT: stride
    """
    embedder = SingleTakensEmbedding(
        parameters_type="fixed",
        time_delay=Tau,
        dimension=dim,
        stride=dT,
        n_jobs=2
    )
    return embedder.fit_transform(x)

# ============================================================
# 6. GENERACIÓN DE LA SERIE TEMPORAL DE HOPF
# ============================================================

def generate_hopf_normal_series(mu, sample_size=SampleSize, fs=fs):
    """
    Genera la serie temporal x(t) del sistema Hopf normal para un valor dado de mu.
    """
    _, ts = simulate_hopf(mu=mu, omega=6.0, L=L, fs=fs, SampleSize=sample_size, InitialConditions=[0.0, 1.01])
    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)
    return np.asarray(ts[0]).ravel()  # Retorna solo x(t)

# ============================================================
# 7. CÁLCULO DE LA MÁXIMA PERSISTENCIA DE H₁
# ============================================================

def compute_max_persistence(point_cloud, max_edge=max_edge):
    """
    Calcula la máxima persistencia para la homología H₁ de una nube de puntos.
    point_cloud: matriz de puntos en el espacio de embedding
    """
    # Crear el objeto de persistencia Vietoris-Rips
    VR = VietorisRipsPersistence(
        homology_dimensions=[1],  # Solo nos interesa H₁ (ciclos)
        metric="manhattan",       # Métrica de Manhattan
        max_edge_length=max_edge   # Distancia máxima para conectar puntos
    )

    # Calcular diagramas de persistencia
    diagrams = VR.fit_transform(point_cloud[None, :, :])
    diag_H1 = diagrams[0][:, :2]  # Extraer (birth, death)

    # Eliminar puntos con muerte infinita
    diag_H1 = diag_H1[np.isfinite(diag_H1[:, 1])]

    # Calcular persistencia (death - birth) y encontrar el máximo
    if diag_H1.shape[0] == 0:
        max_persistence = 0.0
    else:
        persistences = diag_H1[:, 1] - diag_H1[:, 0]
        max_persistence = np.max(persistences)

    return max_persistence

# ============================================================
# 8. CÁLCULO PARA CADA VALOR DE MU
# ============================================================

def compute_max_persistence_for_mu(mu, dim=dim, Tau=tau, dT=dT):
    """
    Calcula la máxima persistencia para un valor específico de mu.
    """
    serie_x = generate_hopf_normal_series(mu)           # Generar serie temporal
    embedding = getTakensEmbedding(serie_x, dim=dim, Tau=Tau, dT=dT)  # Embedding
    return compute_max_persistence(embedding)            # Calcular max persistence

# ============================================================
# 9. FUNCIÓN PRINCIPAL: GRÁFICA Y EXPORTACIÓN
# ============================================================

def plot_max_persistence(mu_values, dim=dim, Tau=tau, dT=dT):
    """
    Calcula y grafica la máxima persistencia para un conjunto de valores de mu.
    """
    mu_vals = []
    max_persistence_vals = []

    # Bucle sobre todos los valores de mu
    for i, mu in enumerate(mu_values):
        mp = compute_max_persistence_for_mu(mu, dim=dim, Tau=Tau, dT=dT)
        max_persistence_vals.append(mp)
        mu_vals.append(mu)
        print(f"μ = {mu:.5f} → max persistence = {mp:.6f} ({i+1}/{len(mu_values)})")

    # ========== CREAR GRÁFICA ==========
    fig, ax = plt.subplots(figsize=(10, 3))

    # Curva de máxima persistencia
    ax.plot(mu_vals, max_persistence_vals, color=CHG_COLOR, lw=1.4)

    # Línea vertical en μ = 0 (punto de bifurcación)
    ax.axvline(0.0, color=DIAG_COLOR, linestyle="--", linewidth=1.0, alpha=0.9)
    ax.text(0.01, 0.96, r"$\mu_c = 0$", transform=ax.get_xaxis_transform(),
            ha="left", va="top", fontsize=9, color="#444444")

    # Configurar ejes
    ax.set_xlim(-1, 1)
    ax.set_xticks(np.arange(-1, 1.01, 0.25))
    ax.margins(x=0.01)
    ax.set_xlabel(r"$\mu$")
    ax.set_ylabel(r"$\max(d-b)$")

    # Aplicar estilo y mostrar
    style_2d_axes(ax)
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()

    # Guardar gráfica como PDF
    fig.savefig(f"figure_hopf_max_persistence_m{dim}_tau{tau}_dT{dT}_noise{ruido}.pdf",
                bbox_inches="tight", dpi=600)
    plt.show()

    # ========== GUARDAR RESULTADOS EN EXCEL ==========
    carpeta = r"D:\MAESTRÍA\TESIS\TAREAS_ASIGNADAS\TAREA_2\TEASPOON PROGRAMAS TDA CROCKER PLOT\15_MÁXIMA PERSISTENCIA\HOPF NORMAL"
    os.makedirs(carpeta, exist_ok=True)  # Crear carpeta si no existe

    archivo_excel = os.path.join(carpeta, f"hopf_max_persistence_{len(mu_values)}_Tau={tau}_m={dim}_noise={ruido}.xlsx")

    df = pd.DataFrame({
        "mu": mu_vals,
        "max_persistence_H1": max_persistence_vals
    })
    df.to_excel(archivo_excel, index=False)

    print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ============================================================
# 10. EJECUCIÓN PRINCIPAL
# ============================================================

if __name__ == "__main__":
    # Valores de mu desde -1 hasta 1 con 300 puntos
    mu_values = np.round(np.linspace(-1, 1, 300), 5)

    # Ejecutar el análisis
    plot_max_persistence(mu_values, dim=dim, Tau=tau, dT=dT)

# Lorenz

Este notebook calcula la máxima persistencia de H₁ para el sistema Lorenz con ρ ∈ [20,30] (300 valores). Usa embedding de Takens (d=2, τ=16, stride=2), Vietoris-Rips con métrica Manhattan y ruido = 0%. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence
import pandas as pd
import os

# ============================================================
# 1. INTEGRADOR RK4 (RUNGE-KUTTA DE 4to ORDEN)
# ============================================================

def rk4_step(f, y, t, dt):
    """Un paso del método RK4."""
    k1 = np.asarray(f(y, t))
    k2 = np.asarray(f(y + dt/2 * k1, t + dt/2))
    k3 = np.asarray(f(y + dt/2 * k2, t + dt/2))
    k4 = np.asarray(f(y + dt * k3, t + dt))
    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

def rk4_integrate(f, y0, t):
    """Integra una EDO usando RK4."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    dt = t[1] - t[0]
    for i in range(1, len(t)):
        y[i] = rk4_step(f, y[i-1], t[i-1], dt)
    return y

# ============================================================
# 2. SIMULADOR DEL SISTEMA DE LORENZ
# ============================================================

L = 100          # Duración total de la simulación
fs = 130         # Frecuencia de muestreo
SampleSize = 2000  # Número de puntos a retener

def simulate_lorenz(rho, sigma=10.0, beta=8/3, L=L, fs=fs, SampleSize=SampleSize, InitialConditions=None):
    """Simula el sistema de Lorenz para un valor dado de rho."""
    if InitialConditions is None:
        InitialConditions = [6.74, 6.74, 22.74]  # Condiciones iniciales [x0, y0, z0]

    t = np.linspace(0, L, int(L * fs))  # Vector de tiempo

    def lorenz_system(state, t):
        """Ecuaciones diferenciales del sistema Lorenz."""
        x, y, z = state
        dxdt = sigma * (y - x)
        dydt = x * (rho - z) - y
        dzdt = x * y - beta * z
        return dxdt, dydt, dzdt

    states = rk4_integrate(lorenz_system, InitialConditions, t)

    # Extraer las series temporales x(t), y(t), z(t)
    ts = [
        states[:, 0][-SampleSize:],  # x(t)
        states[:, 1][-SampleSize:],  # y(t)
        states[:, 2][-SampleSize:]   # z(t)
    ]
    t = t[-SampleSize:]

    return t, ts

# ============================================================
# 3. ESTILO DE LAS GRÁFICAS
# ============================================================

REF_COLOR = "#1F3FFF"   # Azul oscuro
CHG_COLOR = "#D62728"   # Rojo oscuro
DIAG_COLOR = "#000000"  # Negro
GRID_COLOR = "#C8C8C8"  # Gris claro
BG_COLOR = "#FFFFFF"    # Blanco

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
})

def style_2d_axes(ax):
    """Aplica un estilo consistente a los ejes 2D."""
    ax.grid(True, alpha=0.22, color=GRID_COLOR, linewidth=0.7)
    for spine in ["left", "bottom", "top", "right"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis="both", length=3, width=0.8)

# ============================================================
# 4. RUIDO CONTROLADO (OPCIONAL)
# ============================================================

ruido = 0.00  # Nivel de ruido (0% = sin ruido)

def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
    """Agrega ruido gaussiano controlado a una serie temporal."""
    if seed is not None:
        np.random.seed(seed)
    ts = np.asarray(ts)
    if mode == "relative":
        scale = noise_level * np.std(ts, axis=-1, keepdims=True)
    else:
        scale = noise_level
    noise = scale * np.random.randn(*ts.shape)
    return ts + noise

# ============================================================
# 5. PARÁMETROS GLOBALES DEL EMBEDDING
# ============================================================

dim = 2      # Dimensión del embedding
tau = 16     # Tiempo de retardo
dT = 2       # Stride (paso entre ventanas)
max_edge = 20  # Distancia máxima para el complejo de Vietoris-Rips

# ============================================================
# 6. EMBEDDING DE TAKENS
# ============================================================

def getTakensEmbedding(x, dim, Tau, dT=dT):
    """Calcula el embedding de Takens usando giotto-tda."""
    embedder = SingleTakensEmbedding(
        parameters_type="fixed",
        time_delay=Tau,
        dimension=dim,
        stride=dT,
        n_jobs=2
    )
    return embedder.fit_transform(x)

# ============================================================
# 7. GENERACIÓN DE LA SERIE TEMPORAL DE LORENZ
# ============================================================

def generate_lorenz_series(rho, sample_size=SampleSize, fs=fs):
    """Genera la serie temporal x(t) del sistema Lorenz para un valor dado de rho."""
    _, ts = simulate_lorenz(
        rho=rho,
        sigma=10.0,
        beta=8/3,
        L=L,
        fs=fs,
        SampleSize=sample_size,
        InitialConditions=[6.74, 6.74, 22.74]
    )
    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)
    return np.asarray(ts[0]).ravel()  # Retorna solo x(t)

# ============================================================
# 8. CÁLCULO DE LA MÁXIMA PERSISTENCIA DE H₁
# ============================================================

def compute_max_persistence(point_cloud, max_edge=max_edge):
    """Calcula la máxima persistencia para la homología H₁ de una nube de puntos."""
    VR = VietorisRipsPersistence(
        homology_dimensions=[1],
        metric="manhattan",
        max_edge_length=max_edge
    )

    diagrams = VR.fit_transform(point_cloud[None, :, :])
    diag_H1 = diagrams[0][:, :2]
    diag_H1 = diag_H1[np.isfinite(diag_H1[:, 1])]

    if diag_H1.shape[0] == 0:
        max_persistence = 0.0
    else:
        persistences = diag_H1[:, 1] - diag_H1[:, 0]
        max_persistence = np.max(persistences)

    return max_persistence

# ============================================================
# 9. CÁLCULO PARA CADA VALOR DE RHO
# ============================================================

def compute_max_persistence_for_rho(rho, dim=dim, Tau=tau, dT=dT):
    """Calcula la máxima persistencia para un valor específico de rho."""
    serie_x = generate_lorenz_series(rho)
    embedding = getTakensEmbedding(serie_x, dim=dim, Tau=Tau, dT=dT)
    return compute_max_persistence(embedding)

# ============================================================
# 10. FUNCIÓN PRINCIPAL: GRÁFICA Y EXPORTACIÓN
# ============================================================

def plot_max_persistence(rho_values, dim=dim, Tau=tau, dT=dT):
    """Calcula y grafica la máxima persistencia para un conjunto de valores de rho."""
    rho_vals = []
    max_persistence_vals = []

    # Bucle sobre todos los valores de rho
    for i, rho in enumerate(rho_values):
        mp = compute_max_persistence_for_rho(rho, dim=dim, Tau=Tau, dT=dT)
        max_persistence_vals.append(mp)
        rho_vals.append(rho)
        print(f"ρ = {rho:.5f} → max persistence = {mp:.6f} ({i+1}/{len(rho_values)})")

    # ========== CREAR GRÁFICA ==========
    fig, ax = plt.subplots(figsize=(10, 3))

    # Curva de máxima persistencia
    ax.plot(rho_vals, max_persistence_vals, color=CHG_COLOR, lw=1.4)

    # Línea vertical en ρ ≈ 24.74 (punto de transición al caos)
    ax.axvline(24.74, color=DIAG_COLOR, linestyle="--", linewidth=1.0, alpha=0.9)
    ax.text(24.8, 0.95, r"$\rho_c \approx 24.74$", transform=ax.get_xaxis_transform(),
            ha="left", va="top", fontsize=9, color="#444444")

    # Configurar ejes
    ax.set_xlim(20, 30)
    ax.set_xticks(np.arange(20, 30.1, 1))
    ax.margins(x=0.01)
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel(r"$\max(d-b)$")

    # Aplicar estilo y mostrar
    style_2d_axes(ax)
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()

    # Guardar gráfica como PDF
    fig.savefig(f"figure_lorenz_max_persistence_m{dim}_tau{tau}_dT{dT}_noise{ruido}.pdf",
                bbox_inches="tight", dpi=600)
    plt.show()

    # ========== GUARDAR RESULTADOS EN EXCEL ==========
    carpeta = r"D:\MAESTRÍA\TESIS\TAREAS_ASIGNADAS\TAREA_2\TEASPOON PROGRAMAS TDA CROCKER PLOT\15_MÁXIMA PERSISTENCIA\LORENZ"
    os.makedirs(carpeta, exist_ok=True)

    archivo_excel = os.path.join(carpeta, f"lorenz_max_persistence_{len(rho_values)}_Tau={tau}_m={dim}_noise={ruido}.xlsx")

    df = pd.DataFrame({
        "rho": rho_vals,
        "max_persistence_H1": max_persistence_vals
    })
    df.to_excel(archivo_excel, index=False)

    print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ============================================================
# 11. EJECUCIÓN PRINCIPAL
# ============================================================

if __name__ == "__main__":
    # Valores de rho desde 20 hasta 30 con 300 puntos
    rho_values = np.round(np.linspace(20, 30, 300), 5)

    # Ejecutar el análisis
    plot_max_persistence(rho_values, dim=dim, Tau=tau, dT=dT)

# Reacción BZ

Este notebook calcula la máxima persistencia de H₁ para el modelo reducido de Belousov-Zhabotinsky con b ∈ [2,5] (300 valores). Usa embedding de Takens (d=2, τ=57, stride=5), Vietoris-Rips con métrica Manhattan y ruido = 0%. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence
import pandas as pd
import os

# ============================================================
# 1. INTEGRADOR RK4 (RUNGE-KUTTA DE 4to ORDEN)
# ============================================================

def rk4_step(f, y, t, dt):
    """Un paso del método RK4."""
    k1 = np.asarray(f(y, t))
    k2 = np.asarray(f(y + dt/2 * k1, t + dt/2))
    k3 = np.asarray(f(y + dt/2 * k2, t + dt/2))
    k4 = np.asarray(f(y + dt * k3, t + dt))
    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

def rk4_integrate(f, y0, t):
    """Integra una EDO usando RK4."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    dt = t[1] - t[0]
    for i in range(1, len(t)):
        y[i] = rk4_step(f, y[i-1], t[i-1], dt)
    return y

# ============================================================
# 2. SIMULADOR DEL SISTEMA DE BELOUSOV-ZHABOTINSKY (BZ)
# ============================================================

L = 80          # Duración total de la simulación
fs = 90         # Frecuencia de muestreo
SampleSize = 4000  # Número de puntos a retener
x0 = 2          # Condición inicial para x
y0 = 7       # Condición inicial para y

def simulate_bz(b, a=10.0, L=L, fs=fs, SampleSize=SampleSize, InitialConditions=None):
    """Simula el sistema de Belousov-Zhabotinsky para un valor dado de b."""
    if InitialConditions is None:
        InitialConditions = [x0, y0]

    t = np.linspace(0, L, int(L * fs))  # Vector de tiempo

    def bz_system(state, t):
        """Ecuaciones diferenciales del sistema BZ (modelo 2D)."""
        x, y = state
        dxdt = a - x - (4*x*y)/(1 + x**2)
        dydt = b*x*(1 - y/(1 + x**2))
        return dxdt, dydt

    states = rk4_integrate(bz_system, InitialConditions, t)

    # Extraer las series temporales x(t) e y(t)
    ts = [
        states[:, 0][-SampleSize:],  # x(t)
        states[:, 1][-SampleSize:]   # y(t)
    ]
    t = t[-SampleSize:]

    return t, ts

# ============================================================
# 3. ESTILO DE LAS GRÁFICAS
# ============================================================

REF_COLOR = "#1F3FFF"   # Azul oscuro
CHG_COLOR = "#D62728"   # Rojo oscuro
DIAG_COLOR = "#000000"  # Negro
GRID_COLOR = "#C8C8C8"  # Gris claro
BG_COLOR = "#FFFFFF"    # Blanco

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
})

def style_2d_axes(ax):
    """Aplica un estilo consistente a los ejes 2D."""
    ax.grid(True, alpha=0.22, color=GRID_COLOR, linewidth=0.7)
    for spine in ["left", "bottom", "top", "right"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis="both", length=3, width=0.8)

# ============================================================
# 4. RUIDO CONTROLADO (OPCIONAL)
# ============================================================

ruido = 0.00  # Nivel de ruido (0% = sin ruido)

def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
    """Agrega ruido gaussiano controlado a una serie temporal."""
    if seed is not None:
        np.random.seed(seed)
    ts = np.asarray(ts)
    if mode == "relative":
        scale = noise_level * np.std(ts, axis=-1, keepdims=True)
    else:
        scale = noise_level
    noise = scale * np.random.randn(*ts.shape)
    return ts + noise

# ============================================================
# 5. PARÁMETROS GLOBALES DEL EMBEDDING
# ============================================================

dim = 2      # Dimensión del embedding
tau = 58     # Tiempo de retardo
dT = 5       # Stride (paso entre ventanas)
max_edge = 40  # Distancia máxima para el complejo de Vietoris-Rips

# ============================================================
# 6. EMBEDDING DE TAKENS
# ============================================================

def getTakensEmbedding(x, dim, Tau, dT=dT):
    """Calcula el embedding de Takens usando giotto-tda."""
    embedder = SingleTakensEmbedding(
        parameters_type="fixed",
        time_delay=Tau,
        dimension=dim,
        stride=dT,
        n_jobs=2
    )
    return embedder.fit_transform(x)

# ============================================================
# 7. GENERACIÓN DE LA SERIE TEMPORAL DE BZ
# ============================================================

def generate_bz_series(b, sample_size=SampleSize, fs=fs):
    """Genera la serie temporal x(t) del sistema BZ para un valor dado de b."""
    _, ts = simulate_bz(
        b=b,
        a=10.0,
        L=L,
        fs=fs,
        SampleSize=sample_size,
        InitialConditions=[x0, y0]
    )
    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)
    return np.asarray(ts[0]).ravel()  # Retorna solo x(t)

# ============================================================
# 8. CÁLCULO DE LA MÁXIMA PERSISTENCIA DE H₁
# ============================================================

def compute_max_persistence(point_cloud, max_edge=max_edge):
    """Calcula la máxima persistencia para la homología H₁ de una nube de puntos."""
    VR = VietorisRipsPersistence(
        homology_dimensions=[1],
        metric="manhattan",
        max_edge_length=max_edge
    )

    diagrams = VR.fit_transform(point_cloud[None, :, :])
    diag_H1 = diagrams[0][:, :2]
    diag_H1 = diag_H1[np.isfinite(diag_H1[:, 1])]

    if diag_H1.shape[0] == 0:
        max_persistence = 0.0
    else:
        persistences = diag_H1[:, 1] - diag_H1[:, 0]
        max_persistence = np.max(persistences)

    return max_persistence

# ============================================================
# 9. CÁLCULO PARA CADA VALOR DE b
# ============================================================

def compute_max_persistence_for_b(b, dim=dim, Tau=tau, dT=dT):
    """Calcula la máxima persistencia para un valor específico de b."""
    serie_x = generate_bz_series(b)
    embedding = getTakensEmbedding(serie_x, dim=dim, Tau=Tau, dT=dT)
    return compute_max_persistence(embedding)

# ============================================================
# 10. FUNCIÓN PRINCIPAL: GRÁFICA Y EXPORTACIÓN
# ============================================================

b_critico = 3.5  # Punto de bifurcación para el sistema BZ

def plot_max_persistence(b_values, dim=dim, Tau=tau, dT=dT):
    """Calcula y grafica la máxima persistencia para un conjunto de valores de b."""
    b_vals = []
    max_persistence_vals = []

    # Bucle sobre todos los valores de b
    for i, b in enumerate(b_values):
        mp = compute_max_persistence_for_b(b, dim=dim, Tau=Tau, dT=dT)
        max_persistence_vals.append(mp)
        b_vals.append(b)
        print(f"b = {b:.5f} → max persistence = {mp:.6f} ({i+1}/{len(b_values)})")

    # ========== CREAR GRÁFICA ==========
    fig, ax = plt.subplots(figsize=(10, 3))

    # Curva de máxima persistencia
    ax.plot(b_vals, max_persistence_vals, color=CHG_COLOR, lw=1.4)

    # Línea vertical en b = 3.5 (punto de bifurcación)
    ax.axvline(b_critico, color=DIAG_COLOR, linestyle="--", linewidth=1.0, alpha=0.9)
    ax.text(b_critico + 0.02, 0.96, rf"$b_c = {b_critico}$",
            transform=ax.get_xaxis_transform(), ha="left", va="top",
            fontsize=9, color="#444444")

    # Configurar ejes
    ax.set_xlim(2, 5)
    ax.set_xticks(np.arange(2, 5.1, 0.5))
    ax.margins(x=0.01)
    ax.set_xlabel(r"$b$")
    ax.set_ylabel(r"$\max(d-b)$")

    # Aplicar estilo y mostrar
    style_2d_axes(ax)
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout()

    # Guardar gráfica como PDF
    fig.savefig(f"figure_bz_max_persistence_m{dim}_tau{tau}_dT{dT}_noise{ruido}.pdf",
                bbox_inches="tight", dpi=600)
    plt.show()

    # ========== GUARDAR RESULTADOS EN EXCEL ==========
    carpeta = r"D:\MAESTRÍA\TESIS\TAREAS_ASIGNADAS\TAREA_2\TEASPOON PROGRAMAS TDA CROCKER PLOT\15_MÁXIMA PERSISTENCIA\REACCIÓN BZ"
    os.makedirs(carpeta, exist_ok=True)

    archivo_excel = os.path.join(carpeta, f"bz_max_persistence_{len(b_values)}_Tau={tau}_m={dim}_noise={ruido}.xlsx")

    df = pd.DataFrame({
        "b": b_vals,
        "max_persistence_H1": max_persistence_vals
    })
    df.to_excel(archivo_excel, index=False)

    print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ============================================================
# 11. EJECUCIÓN PRINCIPAL
# ============================================================

if __name__ == "__main__":
    # Valores de b desde 2 hasta 5 con 300 puntos
    b_values = np.round(np.linspace(2, 5, 300), 5)

    # Ejecutar el análisis
    plot_max_persistence(b_values, dim=dim, Tau=tau, dT=dT)